<a href="https://colab.research.google.com/github/mtchka/tsukuten_caption_maker/blob/main/tsukuten_caption_maker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# つくてんキャプションメーカー
筑波大学天文研究会の、雙峰祭での写真展などで制作するキャプションのデータをつくるプログラムです。

上の欄の「▶すべてのセルを実行」を押し、順に説明を読んでみてください。

## (1) ライブラリのインストール・インポート


必要な拡張機能的なのを入れています。
実行が終了して✓が付いていたら次に進んでください。

In [ ]:
!pip install python-pptx

from pptx import Presentation
from pptx.util import Pt, Cm
from pptx.enum.text import PP_ALIGN
from pptx.enum.shapes import MSO_CONNECTOR
from pptx.dml.color import RGBColor

from google.colab import files

import csv

## (2) ファイルのアップロード
以下の手順に従ってファイルをアップしてください。
1. Googleフォームのスプレッドシートを開き、「ファイル」→「ダウンロード」→「カンマ区切り形式（.csv）」を選択
1. ダウンロードしたファイルを、以下の「ファイル選択」から選択

In [ ]:
# ファイルをアップロードする
uploaded = files.upload()

filename = list(uploaded.keys())[0]

## (3) コードを実行
操作不要 完成したパワポファイルが勝手にパソコンにダウンロードされます。

たぶんエラー起きないと思いますが起きたら作成者に連絡ください。
### コード変更が必要な場合
不具合等が発生した場合、自分で変えられるようだったら上の欄の「ファイル」→「ドライブにコピーを保存」してからコピーされた方を書き換えてください。主には★で示した部分を変えることになると思います。わからないことがあれば作成者に問い合わせてください。

In [ ]:
slide_width =  Cm(29.7)
slide_height = Cm(21.0)

# ★フォント変えたかったら ' ' の中身を書き換える
# 現在のフォントは無料のZen Maru Gothic
font = 'Zen Maru Gothic'


class TextInfo:
    def __init__(self, left, top, width, alignment, wrap, pt):
        self.left = left
        self.top = top
        self.width = width
        self.alignment = alignment
        self.wrap = wrap
        self.pt = pt


# テキストボックスのx座標, y座標, 幅, 高さ, 文字配置, 文字折り返し, 文字サイズ
text_info_title = TextInfo((slide_width-Cm(10))/2, Cm(2), Cm(10), 'center', False, 66)
text_info_camera = TextInfo(Cm(2.5), Cm(7.5), Cm(4), 'right', False, 26)
text_info_contents = TextInfo(Cm(6),Cm(7.5), slide_width-Cm(9), 'left', True, 26)
text_info_name = TextInfo(slide_width-Cm(12), slide_height-Cm(2.5), Cm(10), 'right', False, 26)


class Caption:
  # ★スプレッドシートの各列を左から 0, 1, 2, ...と数えたときの列番号と各項目を対応させています。
  # （たとえば投稿者名が左から3列目ならself.name = row[2]になる）
  # スプレッドシートでの列対応が異なる場合は []内の数字を入れ替える必要がある

    def __init__(self, row):
        self.name = row[2] # 投稿者名
        self.grade = row[3] # 〇期
        self.title = row[4] # 写真タイトル
        self.explanation = row[5] # 説明文
        self.place = row[6] # 撮影場所
        self.date = row[7] # 撮影日
        self.time = row[8] # 撮影時刻
        self.camera = row[9] # カメラ
        self.telescope = row[10] # 鏡筒
        self.exposure = row[11] # 露出
        self.iso = row[12] # ISO値
        self.f = row[13] # f値
        self.stack = row[14] # スタック枚数


# タイトルと説明の間に線を引くだけの関数
def line_maker(slide):
    line = slide.shapes.add_connector(
        MSO_CONNECTOR.STRAIGHT,
        Cm(2), Cm(6), slide_width-Cm(2), Cm(6)
    )
    line.line.color.rgb = RGBColor(0, 0, 0)
    line.shadow.inherit = False

# テキストを配置する関数
# 引数は(スライド, テキスト内容, テキスト配置情報)
def text_maker(slide, text, text_info):
    textbox = slide.shapes.add_textbox(text_info.left, text_info.top, text_info.width, Cm(10))
    frame = textbox.text_frame
    p = frame.paragraphs[0]
    run = p.add_run()
    run.font.name = font
    run.font.size = Pt(text_info.pt)
    run.text = text
    frame.word_wrap = text_info.wrap

    if text_info.alignment == 'center':
        p.alignment = PP_ALIGN.CENTER
    elif text_info.alignment == 'right':
        p.alignment = PP_ALIGN.RIGHT


# なんかいい感じに文字配置する関数（力技だよ）
# 引数は(スライド, キャプション情報)
def caption_maker(slide, caption):
    text_title = f'{caption.title}'
    text_maker(slide, text_title, text_info_title)

    label_camera = f'\n\n\nカメラ：'
    if caption.telescope != '':
        label_camera += '\n鏡筒：'
    text_maker(slide, label_camera, text_info_camera)

    photo_info = ''
    if caption.exposure != '':
        text_exposure = f'露光{caption.exposure}s'
        photo_info += f'{text_exposure:<10}'
    if caption.iso != '':
        text_iso = f'ISO{caption.iso}'
        photo_info += f'{text_iso:<11}'
    if caption.f != '':
        text_f = f'f{caption.f}'
        photo_info += f'{text_f:<14}'
    if caption.stack != '':
        photo_info += f'{caption.stack}枚スタック'

    text_contents = f'{caption.date} {caption.time}\n{caption.place}\n\n\
{caption.camera}\n{caption.telescope}\n{photo_info}\n\n{caption.explanation}\n'
    text_maker(slide, text_contents, text_info_contents)

    text_name = f'{caption.grade}　{caption.name}'
    text_maker(slide, text_name, text_info_name)


# 白紙パワポを作成➡キャプションデータ読み込み
# ➡キャプションごとにページを追加し、そこにキャプション情報を配置
# ➡全キャプションできたらパワポとして保存 という流れのmain関数
def main():
    prs = Presentation()
    prs.slide_width = slide_width
    prs.slide_height = slide_height


    with open(filename, "r", encoding="utf-8") as f:
        reader = csv.reader(f)

        for row in reader:
            caption = Caption(row)

            slide = prs.slides.add_slide(prs.slide_layouts[6])
            line_maker(slide)
            caption_maker(slide, caption)

    prs.save("captions.pptx")
    files.download("captions.pptx")


if __name__ == '__main__':
    main()